# Recurrent Neural Networks, LSTM, and GRU

**Learning Objectives:** Understand RNNs for sequential data, LSTM gates, GRU architecture, implement sentiment analysis and time series forecasting with Keras.

## 1. RNN Concept

**Recurrent Neural Networks** process sequential data by maintaining a **hidden state** that carries information across time steps.

$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$

$y_t = W_{hy} h_t + b_y$

Unlike feedforward networks, RNNs share weights across time steps, making them efficient for variable-length sequences.

In [ ]:
# Simple RNN forward pass from scratch
import numpy as np

def rnn_forward(X, Wxh, Whh, Why, bh, by):
    """X: (timesteps, input_dim)"""
    h = np.zeros(Whh.shape[0])  # initial hidden state
    outputs = []
    for t in range(len(X)):
        h = np.tanh(Wxh @ X[t] + Whh @ h + bh)
        y = Why @ h + by
        outputs.append(y)
    return np.array(outputs), h

# Demo
np.random.seed(42)
X_demo = np.random.randn(5, 10)  # 5 timesteps, 10 features
Wxh = np.random.randn(20, 10) * 0.01
Whh = np.random.randn(20, 20) * 0.01
Why = np.random.randn(5, 20) * 0.01
bh = np.zeros(20)
by = np.zeros(5)

out, final_h = rnn_forward(X_demo, Wxh, Whh, Why, bh, by)
print(f"Output shape: {out.shape}, Hidden shape: {final_h.shape}")

## 2. Backpropagation Through Time (BPTT)

BPTT unrolls the RNN through time and applies standard backpropagation. The loss gradient at each time step depends on all previous time steps.

**Problem:** Gradients are multiplied by $W_{hh}$ at each step → **vanishing gradient** for long sequences (or exploding if eigenvalues > 1).

This is why vanilla RNNs struggle with long-term dependencies.

## 3. LSTM — Long Short-Term Memory

LSTM (Hochreiter & Schmidhuber, 1997) solves the vanishing gradient problem with a **cell state** and **gates**.

### Gates:
| Gate | Formula | Purpose |
|------|---------|---------|
| **Forget** | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | What to discard from cell state |
| **Input** | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | What new info to store |
| **Candidate** | $\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$ | New candidate values |
| **Output** | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | What to output based on cell state |

### State updates:
- $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$
- $h_t = o_t \odot \tanh(C_t)$

In [ ]:
# LSTM forward pass from scratch
def lstm_forward(X, params):
    """Simplified single-layer LSTM forward pass."""
    Wf, Wi, Wc, Wo, bf, bi, bc, bo = params
    n_x = X.shape[1]
    n_h = Wf.shape[0] // 2

    h = np.zeros(n_h)
    C = np.zeros(n_h)
    outputs = []

    for t in range(len(X)):
        concat = np.concatenate([h, X[t]])
        f = 1 / (1 + np.exp(-(Wf @ concat + bf)))  # forget gate
        i = 1 / (1 + np.exp(-(Wi @ concat + bi)))  # input gate
        C_candidate = np.tanh(Wc @ concat + bc)
        o = 1 / (1 + np.exp(-(Wo @ concat + bo)))  # output gate

        C = f * C + i * C_candidate
        h = o * np.tanh(C)
        outputs.append(h)

    return np.array(outputs), (h, C)

# Quick sanity check
n_hidden = 16
n_input = 10
W = [np.random.randn(n_hidden, n_hidden + n_input) * 0.01 for _ in range(4)]
b = [np.zeros(n_hidden) for _ in range(4)]

X_test = np.random.randn(8, n_input)
out, state = lstm_forward(X_test, W + b)
print(f"LSTM output shape: {out.shape}")

## 4. GRU — Gated Recurrent Unit

GRU (Cho et al., 2014) simplifies LSTM by merging cell state and hidden state, using only two gates:

| Gate | Formula | Purpose |
|------|---------|---------|
| **Reset** | $r_t = \sigma(W_r [h_{t-1}, x_t])$ | How much past to forget |
| **Update** | $z_t = \sigma(W_z [h_{t-1}, x_t])$ | How much to update hidden state |

$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t])$

$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

GRU has fewer parameters than LSTM and often performs similarly.

## 5. LSTM & GRU in Keras — Sentiment Analysis

We'll use the IMDB reviews dataset for binary sentiment classification.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load IMDB dataset (word indices)
vocab_size = 20000
max_len = 200

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)

# Pad sequences to same length
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=max_len)
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=max_len)

print(f"Train: {x_train.shape}, Test: {x_test.shape}")

In [ ]:
# Build an LSTM model for sentiment analysis
lstm_model = keras.Sequential([
    layers.Embedding(vocab_size, 128, input_length=max_len),
    layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

In [ ]:
# Train the LSTM model
history = lstm_model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2,
    verbose=1
)

test_loss, test_acc = lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

In [ ]:
# Decode IMDB review for inspection
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {v: k for k, v in word_index.items()}

def decode_review(sequence):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in sequence if i > 0])

# Predict sentiment
idx = 42
pred = lstm_model.predict(x_test[idx:idx+1])[0][0]
sentiment = 'Positive' if pred > 0.5 else 'Negative'
print(f"Prediction: {pred:.3f} → {sentiment}")
print(f"True label: {'Positive' if y_test[idx] else 'Negative'}")
print(f"Review preview: {decode_review(x_test[idx])[:200]}...")

## 6. GRU Model for Comparison

In [ ]:
# Build a GRU model
gru_model = keras.Sequential([
    layers.Embedding(vocab_size, 128, input_length=max_len),
    layers.GRU(64, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(1, activation='sigmoid')
])

gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_gru = gru_model.fit(
    x_train, y_train,
    batch_size=64, epochs=5,
    validation_split=0.2, verbose=0
)

gru_test_acc = gru_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"GRU test accuracy: {gru_test_acc:.4f}")

## 7. Bidirectional RNNs

Bidirectional RNNs process sequences in both forward and backward directions, capturing context from both past and future.

In [ ]:
# Bidirectional LSTM
bi_lstm_model = keras.Sequential([
    layers.Embedding(vocab_size, 128, input_length=max_len),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    layers.Dense(1, activation='sigmoid')
])

bi_lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
bi_lstm_model.summary()

## 8. Time Series Forecasting with LSTM

LSTMs excel at time series prediction (weather, stock, energy).

In [ ]:
# Generate synthetic time series data
def generate_time_series(n_steps=1000):
    t = np.linspace(0, 20 * np.pi, n_steps)
    series = np.sin(t) + 0.3 * np.sin(3 * t) + 0.1 * np.random.randn(n_steps)
    return series

series = generate_time_series(2000)
plt.figure(figsize=(12, 3))
plt.plot(series[:500])
plt.title('Synthetic Time Series')
plt.show()

# Create sequences for supervised learning
def create_sequences(data, seq_length=50):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(series, 50)
X_seq = X_seq.reshape(-1, 50, 1)  # (samples, timesteps, features)

split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

In [ ]:
# LSTM for time series forecasting
ts_model = keras.Sequential([
    layers.LSTM(32, activation='relu', input_shape=(50, 1)),
    layers.Dense(1)
])

ts_model.compile(optimizer='adam', loss='mse')
ts_model.fit(X_train, y_train, batch_size=32, epochs=10, validation_split=0.1, verbose=1)

# Predict and visualize
y_pred = ts_model.predict(X_test)

plt.figure(figsize=(12, 4))
plt.plot(y_test[:200], label='True')
plt.plot(y_pred[:200], label='Predicted')
plt.title('Time Series Forecasting')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Interview Problems

### Problem 1: Sentiment Analysis on IMDB

Build an LSTM model with dropout and embedding for IMDB sentiment. Achieve >85% test accuracy. Compare with a simple RNN baseline.

### Problem 2: Time Series Forecasting with LSTM

Use LSTM to forecast the next 10 steps of a sine wave sequence (multi-step prediction). Use teacher forcing during training.

### Problem 3: Implement Gradient Clipping for RNN

Explain why RNNs need gradient clipping more than CNNs. Implement a function to clip gradients by global norm.

In [ ]:
# Solution: Problem 2 — Multi-step forecasting
def create_multi_step_data(data, seq_in=50, seq_out=10):
    X, y = [], []
    for i in range(len(data) - seq_in - seq_out + 1):
        X.append(data[i:i + seq_in])
        y.append(data[i + seq_in:i + seq_in + seq_out])
    return np.array(X), np.array(y)

X_multi, y_multi = create_multi_step_data(series, 50, 10)
X_multi = X_multi.reshape(-1, 50, 1)

split = int(0.8 * len(X_multi))
X_train_m, y_train_m = X_multi[:split], y_multi[:split]
X_test_m, y_test_m = X_multi[split:], y_multi[split:]

multi_model = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(50, 1)),
    layers.LSTM(32, return_sequences=False),
    layers.Dense(10)
])
multi_model.compile(optimizer='adam', loss='mse')
multi_model.fit(X_train_m, y_train_m, batch_size=32, epochs=10, verbose=0)

# Visualize one prediction
idx = 0
pred_seq = multi_model.predict(X_test_m[idx:idx+1])[0]
true_seq = y_test_m[idx]

plt.figure(figsize=(10, 3))
plt.plot(true_seq, 'b-', label='True')
plt.plot(pred_seq, 'r--', label='Predicted')
plt.title('10-Step Forecast')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Summary

- **RNNs** process sequential data with shared weights and hidden state
- **BPTT** unrolls the RNN for gradient computation but suffers from vanishing gradients
- **LSTM** uses forget, input, output gates and cell state to capture long-range dependencies
- **GRU** simplifies LSTM with reset and update gates (fewer parameters)
- **Bidirectional RNNs** process sequences in both directions for richer context
- **Embedding layers** map discrete tokens to dense vector representations
- **Keras layers**: `SimpleRNN`, `LSTM`, `GRU`, `Bidirectional`, `Embedding`
- Applications: sentiment analysis, time series forecasting, language modeling, text generation